# 基于深度学习的家具识别系统
## 《深度学习应用开发》课程设计

**选题**: 基于深度学习的家具识别系统  
**类别**: chair(椅子), table(桌子), sofa(沙发), bed(床), cabinet(柜子)  
**框架**: PyTorch + MobileNetV2 迁移学习

## 1. 导入所需的库与模块

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import MobileNet_V2_Weights

import matplotlib
matplotlib.use('Agg')  # 非交互式后端，用于生成截图
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from collections import Counter
import os, random, time
from PIL import Image

# 中文显示设置
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 固定随机种子确保可复现
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# 设备选择
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')
print(f'PyTorch 版本: {torch.__version__}')

使用设备: cpu
PyTorch 版本: 2.11.0+cpu


## 2. 数据准备及数据预处理

### 2.1 数据集路径配置

In [2]:
# 数据集路径
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'dataset')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'val')
TEST_DIR = os.path.join(DATA_DIR, 'test')

# 如果上述路径不存在，尝试使用当前目录
if not os.path.exists(DATA_DIR):
    DATA_DIR = 'dataset'
    TRAIN_DIR = os.path.join(DATA_DIR, 'train')
    VAL_DIR = os.path.join(DATA_DIR, 'val')
    TEST_DIR = os.path.join(DATA_DIR, 'test')

print(f'数据集路径: {DATA_DIR}')
print(f'训练集: {TRAIN_DIR}')
print(f'验证集: {VAL_DIR}')
print(f'测试集: {TEST_DIR}')

数据集路径: C:\Users\zzz\Desktop\家具分类\dataset
训练集: C:\Users\zzz\Desktop\家具分类\dataset\train
验证集: C:\Users\zzz\Desktop\家具分类\dataset\val
测试集: C:\Users\zzz\Desktop\家具分类\dataset\test


### 2.2 数据增强与预处理

In [3]:
# 图像参数
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 5

# ImageNet 均值与标准差（MobileNetV2 预训练使用）
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# 训练集数据增强
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# 验证集/测试集预处理（不做增强）
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

print('训练集增强: RandomHorizontalFlip, RandomRotation(20), ColorJitter, RandomAffine')
print(f'输入尺寸: {IMG_SIZE}x{IMG_SIZE}')
print(f'批次大小: {BATCH_SIZE}')

训练集增强: RandomHorizontalFlip, RandomRotation(20), ColorJitter, RandomAffine
输入尺寸: 224x224
批次大小: 32


### 2.3 数据加载

In [4]:
# 加载数据集
try:
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)
    test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
except Exception as e:
    print(f'直接加载失败: {e}')
    print('尝试从单一目录划分...')
    full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
    n_total = len(full_dataset)
    n_train = int(n_total * 0.7)
    n_val = int(n_total * 0.15)
    n_test = n_total - n_train - n_val
    train_dataset, val_dataset, test_dataset = random_split(
        full_dataset, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)
    )

# 创建 DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# 类别映射
if hasattr(train_dataset, 'classes'):
    class_names = train_dataset.classes
elif hasattr(full_dataset, 'classes'):
    class_names = full_dataset.classes
else:
    class_names = ['bed', 'cabinet', 'chair', 'sofa', 'table']

class_to_idx = {name: i for i, name in enumerate(class_names)}
print(f'类别 ({len(class_names)} 类): {class_names}')
print(f'类别映射: {class_to_idx}')
print(f'\n数据集统计:')
print(f'  训练集: {len(train_dataset)} 张')
print(f'  验证集: {len(val_dataset)} 张')
print(f'  测试集: {len(test_dataset)} 张')
print(f'  总计: {len(train_dataset) + len(val_dataset) + len(test_dataset)} 张')

类别 (5 类): ['bed', 'cabinet', 'chair', 'sofa', 'table']
类别映射: {'bed': 0, 'cabinet': 1, 'chair': 2, 'sofa': 3, 'table': 4}

数据集统计:
  训练集: 2100 张
  验证集: 450 张
  测试集: 450 张
  总计: 3000 张


### 2.4 数据可视化

In [5]:
# 统计各类别样本数
all_labels = []
for _, label in train_dataset:
    all_labels.append(label)

class_counts = Counter(all_labels)
print('各类别训练样本数:')
for cls_name in class_names:
    idx = class_to_idx[cls_name]
    print(f'  {cls_name}: {class_counts[idx]} 张')

各类别训练样本数:
  bed: 420 张
  cabinet: 420 张
  chair: 420 张
  sofa: 420 张
  table: 420 张


In [6]:
# 类别分布柱状图
fig, ax = plt.subplots(figsize=(10, 5))
counts = [class_counts[class_to_idx[name]] for name in class_names]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
bars = ax.bar(class_names, counts, color=colors)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(count),
            ha='center', fontsize=12, fontweight='bold')
ax.set_title('训练集各类别样本分布', fontsize=16, fontweight='bold')
ax.set_ylabel('样本数量', fontsize=12)
ax.set_ylim(0, max(counts) * 1.2)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\zzz\AppData\Local\Temp\ipykernel_19984\2319558593.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# 展示各类别样本图片
def denormalize(tensor):
    """反归一化以便显示"""
    img = tensor.clone()
    for t, m, s in zip(img, MEAN, STD):
        t.mul_(s).add_(m)
    return img.clamp(0, 1)

fig, axes = plt.subplots(len(class_names), 6, figsize=(15, 3 * len(class_names)))
for i, cls_name in enumerate(class_names):
    cls_idx = class_to_idx[cls_name]
    # 找该类别的前 6 张图片
    count = 0
    for img, label in train_dataset:
        if label == cls_idx and count < 6:
            ax = axes[i, count]
            img_display = denormalize(img)
            ax.imshow(img_display.permute(1, 2, 0))
            ax.set_xticks([])
            ax.set_yticks([])
            if count == 0:
                ax.set_ylabel(cls_name, fontsize=12, fontweight='bold')
            count += 1
        if count >= 6:
            break
fig.suptitle('各类别训练样本展示', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\zzz\AppData\Local\Temp\ipykernel_19984\872618827.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. 模型搭建与参数优化

### 3.1 迁移学习 — MobileNetV2

In [8]:
def build_model(num_classes=5, freeze_backbone=True):
    """
    基于 MobileNetV2 的迁移学习模型
    - 使用 ImageNet 预训练权重
    - 冻结骨干网络，仅训练分类头
    - 添加 Dropout 防止过拟合
    """
    # 加载预训练 MobileNetV2
    model = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)

    # 冻结特征提取层
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
        print('已冻结 MobileNetV2 骨干网络')

    # 替换分类头
    in_features = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )

    return model

model = build_model(num_classes=NUM_CLASSES, freeze_backbone=False)
model = model.to(device)
print(f'模型已创建，参数总数: {sum(p.numel() for p in model.parameters()):,}')
print(f'可训练参数: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

模型已创建，参数总数: 2,882,309
可训练参数: 2,882,309


### 3.2 模型结构显示

In [9]:
print('=' * 70)
print('模型结构 — MobileNetV2 + 自定义分类头')
print('=' * 70)
print(model)
print('\n' + '=' * 70)
print('分类头结构:')
print(model.classifier)

模型结构 — MobileNetV2 + 自定义分类头
MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
       

### 3.3 模型参数设置

In [10]:
# 损失函数与优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)

# 学习率调度器：验证集 loss 不下降时降低学习率
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6
)

print('优化器: Adam')
print(f'初始学习率: {optimizer.param_groups[0]["lr"]}')
print(f'权重衰减: 1e-4')
print('学习率调度: ReduceLROnPlateau (factor=0.5, patience=3)')
print('损失函数: CrossEntropyLoss')

优化器: Adam
初始学习率: 0.0005
权重衰减: 1e-4
学习率调度: ReduceLROnPlateau (factor=0.5, patience=3)
损失函数: CrossEntropyLoss


## 4. 模型编译与训练

In [11]:
def train_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    """在验证集/测试集上评估"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [12]:
# 训练参数
NUM_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 7

print(f'训练轮数: {NUM_EPOCHS}')
print(f'早停耐心值: {EARLY_STOPPING_PATIENCE}')
print('=' * 60)

# 记录训练历史
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

best_val_acc = 0.0
best_epoch = 0
patience_counter = 0
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    # 训练
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    # 验证
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    # 记录
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # 学习率调整
    scheduler.step(val_loss)

    elapsed = time.time() - epoch_start
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | '
          f'LR: {optimizer.param_groups[0]["lr"]:.2e} | Time: {elapsed:.1f}s')

    # 保存最佳模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'class_names': class_names,
            'class_to_idx': class_to_idx,
        }, 'furniture_model.pth')
    else:
        patience_counter += 1

    # 早停检查
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f'\n早停触发！验证准确率 {EARLY_STOPPING_PATIENCE} 个 epoch 未提升。')
        break

total_time = time.time() - start_time
print(f'\n训练完成！总耗时: {total_time:.0f}s ({total_time/60:.1f}min)')
print(f'最佳验证准确率: {best_val_acc:.4f} (Epoch {best_epoch})')

训练轮数: 30
早停耐心值: 7
Epoch   1/30 | Train Loss: 0.9522 | Train Acc: 0.5990 | Val Loss: 0.6178 | Val Acc: 0.7667 | LR: 5.00e-04 | Time: 126.6s
Epoch   2/30 | Train Loss: 0.5953 | Train Acc: 0.7810 | Val Loss: 0.4753 | Val Acc: 0.8378 | LR: 5.00e-04 | Time: 132.3s
Epoch   3/30 | Train Loss: 0.4860 | Train Acc: 0.8310 | Val Loss: 0.5918 | Val Acc: 0.8022 | LR: 5.00e-04 | Time: 143.2s
Epoch   4/30 | Train Loss: 0.4576 | Train Acc: 0.8357 | Val Loss: 0.5024 | Val Acc: 0.8244 | LR: 5.00e-04 | Time: 149.9s
Epoch   5/30 | Train Loss: 0.3740 | Train Acc: 0.8657 | Val Loss: 0.5370 | Val Acc: 0.8289 | LR: 5.00e-04 | Time: 176.3s
Epoch   6/30 | Train Loss: 0.3841 | Train Acc: 0.8662 | Val Loss: 0.3917 | Val Acc: 0.8667 | LR: 5.00e-04 | Time: 207.5s
Epoch   7/30 | Train Loss: 0.3200 | Train Acc: 0.8876 | Val Loss: 0.4357 | Val Acc: 0.8600 | LR: 5.00e-04 | Time: 142.2s
Epoch   8/30 | Train Loss: 0.2751 | Train Acc: 0.9119 | Val Loss: 0.4838 | Val Acc: 0.8356 | LR: 5.00e-04 | Time: 141.3s
Epoch   9/30 |

### 4.1 模型训练过程可视化 — 准确率及 Loss 曲线

In [13]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss 曲线
ax1.plot(epochs_range, history['train_loss'], 'b-', label='训练集 Loss', linewidth=2)
ax1.plot(epochs_range, history['val_loss'], 'r-', label='验证集 Loss', linewidth=2)
ax1.scatter(best_epoch, history['val_loss'][best_epoch - 1],
            color='red', s=100, zorder=5, label=f'最佳 Epoch ({best_epoch})')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('训练 Loss 与验证 Loss 曲线', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy 曲线
ax2.plot(epochs_range, history['train_acc'], 'b-', label='训练集准确率', linewidth=2)
ax2.plot(epochs_range, history['val_acc'], 'r-', label='验证集准确率', linewidth=2)
ax2.scatter(best_epoch, best_val_acc,
            color='red', s=100, zorder=5, label=f'最佳 ({best_val_acc:.2%})')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('训练准确率与验证准确率曲线', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\zzz\AppData\Local\Temp\ipykernel_19984\1807417272.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. 模型评估与预测

### 5.1 加载最佳模型

In [14]:
# 加载训练好的最佳模型权重
checkpoint = torch.load('furniture_model.pth', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"已加载模型 (Epoch {checkpoint['epoch']}, Val Acc: {checkpoint['val_acc']:.4f})")

已加载模型 (Epoch 18, Val Acc: 0.9022)


### 5.2 测试集评估 — 模型精度分析

In [15]:
# 在测试集上评估
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'\n测试集结果:')
print(f'  Loss:     {test_loss:.4f}')
print(f'  Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

# 准确率评定
if test_acc >= 0.95:
    grade = '优秀 (≥95%)'
elif test_acc >= 0.85:
    grade = '良好 (≥85%)'
elif test_acc >= 0.75:
    grade = '合格 (≥75%)'
else:
    grade = '待改进 (<75%)'
print(f'  评定: {grade}')


测试集结果:
  Loss:     0.4598
  Accuracy: 0.8933 (89.33%)
  评定: 良好 (≥85%)


### 5.3 混淆矩阵

In [16]:
# 收集测试集所有预测结果
all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# 混淆矩阵
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('预测类别', fontsize=12)
ax.set_ylabel('真实类别', fontsize=12)
ax.set_title('测试集混淆矩阵', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\zzz\AppData\Local\Temp\ipykernel_19984\1421347081.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# 分类报告
print('=' * 60)
print('分类报告 (Classification Report)')
print('=' * 60)
report = classification_report(all_labels, all_preds, target_names=class_names, digits=4)
print(report)

分类报告 (Classification Report)
              precision    recall  f1-score   support

         bed     0.8400    0.9333    0.8842        90
     cabinet     0.9773    0.9556    0.9663        90
       chair     0.9149    0.9556    0.9348        90
        sofa     0.8625    0.7667    0.8118        90
       table     0.8750    0.8556    0.8652        90

    accuracy                         0.8933       450
   macro avg     0.8939    0.8933    0.8924       450
weighted avg     0.8939    0.8933    0.8924       450



### 5.4 预测结果展示

In [18]:
# 从测试集随机抽样展示预测结果
def predict_single_image(model, image_tensor, device):
    """单张图片预测"""
    model.eval()
    with torch.no_grad():
        image_tensor = image_tensor.unsqueeze(0).to(device)
        outputs = model(image_tensor)
        probs = torch.softmax(outputs, dim=1)
        _, pred = torch.max(outputs, 1)
    return pred.item(), probs.squeeze().cpu().numpy()

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()

# 每类随机选 2 张测试图片
samples_per_class = 2
shown = []
for cls_idx, cls_name in enumerate(class_names):
    cls_images = []
    for img, label in test_dataset:
        if label == cls_idx:
            cls_images.append(img)
    random.shuffle(cls_images)
    for j in range(samples_per_class):
        if j < len(cls_images):
            shown.append((cls_images[j], cls_idx, cls_name))

for i, (img, true_idx, true_name) in enumerate(shown[:10]):
    pred_idx, probs = predict_single_image(model, img, device)
    pred_name = class_names[pred_idx]
    confidence = probs[pred_idx]

    img_display = denormalize(img)
    ax = axes[i]
    ax.imshow(img_display.permute(1, 2, 0))
    color = 'green' if pred_idx == true_idx else 'red'
    ax.set_title(f'真实: {true_name}\n预测: {pred_name} ({confidence:.1%})',
                 color=color, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('测试集预测结果展示 (绿色=正确, 红色=错误)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('prediction_samples.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\zzz\AppData\Local\Temp\ipykernel_19984\3828464444.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. 结果分析与展示

本项目基于 MobileNetV2 迁移学习实现了 5 类家具（椅子、桌子、沙发、床、柜子）的自动识别与分类。

### 技术总结
- **数据集**: CIFAR-100 家具图像数据集（从 CIFAR-100 提取 5 类真实家具照片），按 70/15/15 划分训练/验证/测试集
- **数据预处理**: Resize(224×224), 随机水平翻转, 旋转, 色彩抖动, ImageNet 标准化
- **模型架构**: MobileNetV2 (ImageNet 预训练) + 自定义分类头 (512 → 5)
- **训练策略**: Adam 优化器, ReduceLROnPlateau 学习率调度, Early Stopping, Dropout 正则化
- **效果**: 见上方的准确率曲线和混淆矩阵

### 前端界面
运行 `app.py` 启动 Flask Web 服务，可通过浏览器上传图片进行识别。
```
python app.py
```
然后访问 http://127.0.0.1:5000